## Car Damage Detection and Segmentation

This notebook implements yolov11 segmentation to detect and isolate types of damage on cars. The dataset comes from https://cardd-ustc.github.io/

### Dependency Imports

In [ ]:
from ultralytics import YOLO
import pandas as pd
import json
import os
import random
import shutil
from IPython.display import Image, display
import glob

### Data Preprocessing

In [ ]:
# print("Formatting dataset...")

# # Create directory structure
# splits = ['train', 'val', 'test']
# for split in splits:
#     os.makedirs(f'../dataset/CarDD_YOLO/{split}/images', exist_ok=True)
#     os.makedirs(f'../dataset/CarDD_YOLO/{split}/labels', exist_ok=True)

#     # Load COCO annotations
#     with open(f'../dataset/CarDD_COCO/annotations/instances_{split}2017.json', 'r') as f:
#         split_data = json.load(f)

#     images = split_data['images']
#     annotations = split_data['annotations']
    
#     # Group annotations by image_id (one image can have multiple annotations)
#     annotations_by_image = {}
#     for annotation in annotations:
#         image_id = annotation['image_id']
#         if image_id not in annotations_by_image:
#             annotations_by_image[image_id] = []
#         annotations_by_image[image_id].append(annotation)
    
#     print(f"\nProcessing {split} split: {len(images)} images, {len(annotations)} annotations")
    
#     for image in images:
#         image_name = image['file_name']
#         image_id = image['id']
#         width = image['width']
#         height = image['height']
        
#         # Copy image to YOLO directory
#         src_path = f'../dataset/CarDD_COCO/{split}2017/{image_name}'
#         dst_path = f'../dataset/CarDD_YOLO/{split}/images/{image_name}'
#         shutil.copy(src_path, dst_path)
        
#         # Create corresponding label file
#         label_name = os.path.splitext(image_name)[0] + '.txt'
#         label_path = f'../dataset/CarDD_YOLO/{split}/labels/{label_name}'
        
#         # Get all annotations for this image
#         image_annotations = annotations_by_image.get(image_id, [])
        
#         with open(label_path, 'w') as label_file:
#             for annotation in image_annotations:
#                 # YOLO uses 0-indexed classes, COCO uses 1-indexed
#                 class_id = annotation['category_id'] - 1
                
#                 # Process each segmentation polygon
#                 for segmentation in annotation['segmentation']:
#                     # Normalize coordinates (YOLO format requires values between 0 and 1)
#                     normalized_coords = []
#                     for i in range(0, len(segmentation), 2):
#                         x = segmentation[i] / width
#                         y = segmentation[i + 1] / height
#                         normalized_coords.extend([x, y])
                    
#                     # Write to label file: class_id x1 y1 x2 y2 ... xn yn
#                     line = f"{class_id} " + " ".join([f"{coord:.6f}" for coord in normalized_coords])
#                     label_file.write(line + '\n')

# print("\nDataset conversion complete!")
# print("\nCategory mapping (YOLO class_id: name):")
# categories = split_data['categories']
# for cat in categories:
#     print(f"  {cat['id'] - 1}: {cat['name']}")

### Create YOLO Dataset Configuration

In [ ]:
# # Create YOLO dataset configuration file
# import yaml

# dataset_config = {
#     'path': '../cardd_dataset/CarDD_YOLO',  # Dataset root directory
#     'train': 'train/images',  # Training images
#     'val': 'val/images',      # Validation images
#     'test': 'test/images',    # Test images
    
#     # Classes
#     'names': {
#         0: 'dent',
#         1: 'scratch',
#         2: 'crack',
#         3: 'glass shatter',
#         4: 'lamp broken',
#         5: 'tire flat'
#     }
# }

# config_path = '../cardd_dataset/CarDD_YOLO/data.yaml'
# with open(config_path, 'w') as f:
#     yaml.dump(dataset_config, f, default_flow_style=False)

# print(f"YOLO dataset configuration saved to: {config_path}")
# print("\nDataset is ready for YOLO segmentation training!")

### Train YOLOv11 Segmentation Model

In [ ]:
model = YOLO('base/yolo11n-seg.pt') 

results = model.train(
    data='../cardd_dataset/CarDD_YOLO/data.yaml',
    epochs=50,
    imgsz=1280,
    batch=16,
    name='car_damage_seg_aug_v1',
    project='runs/segment',
    patience=10,
    save=True,
    plots=True,

    # Heavy Augmentations
    degrees=5.0,
    translate=0.05,
    scale=0.15,
    shear=2.0,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.2,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.3,
    mixup=0.05,
    copy_paste=0.2,
    erasing=0.15,
    auto_augment=None
)

### Evaluate Model Performance

In [ ]:
# No Augmentations
best_model = YOLO('runs/segment/car_damage_seg_no_aug/weights/best.pt')
metrics = best_model.val(data='../cardd_dataset/CarDD_YOLO/data.yaml', split='test')

print("\nTest Set Results:")
print(f"mAP50: {metrics.seg.map50:.4f}")
print(f"mAP50-95: {metrics.seg.map:.4f}")

# Heavy Augmentations
best_model = YOLO('runs/segment/car_damage_seg_aug_v16/weights/best.pt')
metrics = best_model.val(data='../cardd_dataset/CarDD_YOLO/data.yaml', split='test')

print("\nTest Set Results:")
print(f"mAP50: {metrics.seg.map50:.4f}")
print(f"mAP50-95: {metrics.seg.map:.4f}")

### Run Inference on Test Images

In [ ]:
test_image_path = '../cardd_dataset/CarDD_YOLO/test/images'

results = best_model.predict(
    source=test_image_path,
    save=True,  # Save annotated images
    conf=0.25,  # Confidence threshold
    project='runs/segment',
    name='predictions',
    exist_ok=True
)

print(f"\nProcessed {len(results)} images")
print(f"Results saved to: runs/segment/predictions")

In [ ]:
pred_images = glob.glob('runs/segment/predictions/*.jpg')
if pred_images:
    print("\nSample prediction:")
    display(Image(filename=pred_images[random.randint(0, len(pred_images))]))